---
image: example.gif
pub-info:
    abstract: |
        A more involved Ciw model, taken from the official Ciw tutorial, animated with a custom
        background image. Shows how vidigi's Ciw helpers scale up to handle more complex
        network-of-queues models.
execute: 
  enabled: true
---

# A More Complex Ciw Example with Backgrounds 

Note that this example is written using ciw 3.x, and is taken from the Ciw documentation here: [https://ciw.readthedocs.io/en/latest/Tutorial/tutorial_iv.html](https://ciw.readthedocs.io/en/latest/Tutorial/tutorial_iv.html)

All credit for Ciw and the Ciw documentation go to Geraint Palmer.

In [ ]:
import pandas as pd
# Import the wrapper objects for model interaction.
from ex_10_ciw_model import N
from vidigi.ciw import event_log_from_ciw_recs
from vidigi.utils import EventPosition, create_event_position_df
from vidigi.animation import animate_activity_log
import plotly.io as pio
pio.renderers.default = "notebook"
import os
import ciw

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "ex_10_ciw_model.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code for the ciw model

```python
{code_content}
```

:::

""")

First, we set the seed and then run the simulation. We then use the `.get_all_records()` method on the simulation object to return the data in the format Vidigi requires.

In [ ]:
ciw.seed(42)

Q = ciw.Simulation(N)

RESULTS_COLLECTION_PERIOD = 180 * 60 # 180 minutes x 60 seconds

Q.simulate_until_max_time(RESULTS_COLLECTION_PERIOD)

recs = Q.get_all_records()

pd.DataFrame(recs).sort_values(['id_number', 'service_start_date']).head(20)


We then use Vidigi's `event_log_from_ciw_recs()` function to convert the Ciw logs into the format the Vidigi animation functions will require. 

This takes the output of `.get_all_records()`, as well as a list of node names that we choose that must match the order of nodes in the simulation.

We'll create our Vidigi event log, and then view the rows that are created for a particular entity.

(If you also want Vidigi's post-run analysis tools, `vidigi.ciw` has `event_logger_from_ciw_recs()` and `trial_logger_from_ciw_recs()`, which return a populated `EventLogger` / `TrialLogger` from the same Ciw records instead of a plain DataFrame.)

In [ ]:
event_log = event_log_from_ciw_recs(
    recs,
    node_name_list=["cold_food", "hot_food", "till"]
    )

event_log[event_log["entity_id"]==13]

So that we can visualise the number of resources available at each step, we need to create a simple class we can pass to the animation function, where the attributes are related to the number of resources.

We'll use these attribute names when we set up our event positioning dataframe shortly. 

In [ ]:
# Create a suitable class to pass in the resource numbers to the animation function
class model_params():
    def __init__(self):
        self.cold_food_servers = 1
        self.hot_food_servers = 2
        self.tills = 2

params = model_params()

We then set up an event positioning dataframe that reflects the **bottom-right** corner of each event. Queues will build up to the left of this point, and then wrap onto a new line when they reach a limit we specify in the animation step, at which point the entity icons will wrap onto a new line, which will appear above (i.e. higher up in the animation) the first row of the queue. 

In [ ]:
event_position_df = create_event_position_df( [
    EventPosition(event='arrival', x=30, y=550, label="Arrival"),
    EventPosition(event='cold_food_wait_begins', x=200, y=510, label="Waiting for Cold Food"),
    EventPosition(event='cold_food_begins', x=210, y=370, resource='cold_food_servers', label="Being Served Cold Food"),
    EventPosition(event='hot_food_wait_begins', x=505, y=510, label="Waiting for Hot Food"),
    EventPosition(event='hot_food_begins', x=505, y=370, resource='hot_food_servers', label="Being Served Hot Food"),
    EventPosition(event='till_wait_begins', x=350, y=170, label="Waiting for Till"),
    EventPosition(event='till_begins', x=350, y=120, resource='tills', label="Being Served at Till"),
    EventPosition(event='depart', x=600, y=10, label="Exit")
]
)

Finally, we create our animation. Here, we've used our all-in-one function as we don't need to do anything special to the entity icons or pathing. 

In [ ]:
animate_activity_log(
    # pass in the log of events we created using the helper function
    event_log=event_log,
    # pass in our initial event positioning dataframe
    event_position_df=event_position_df,
    # pass in our model parameters class so that resources have icons that persist when not in use
    scenario=model_params(),
    # this sim was set up in seconds - vidigi defaults to minutes, so its important we overwrite this
    simulation_time_unit="seconds",
    # display the time below the bar in the format 'Day Month Year Hours:Minutes:Seconds'.
    # because we don't specify a start date for the animation to run from, it will default to 6
    # months from when the animation is created
    time_display_units="dhms",
    # print out messages to help us spot where any possible issues occur in the animation
    # generation process
    debug_mode=True,
    # hide gridlines and axis labels
    setup_mode=False,
    # poll the position of entities every 5 seconds
    every_x_time_units=5,
    # Include a play button (so the viewer can click play and leave the animation to run, rather
    # than only having the option to scrub through)
    include_play_button=True,
    # set icon to 20 units in size
    entity_icon_size=20,
    # set the horizontal gap between entities to be 25 units in size
    gap_between_entities=15,
    # set the vertical gap between rows of entities when queueing to be 25 units
    gap_between_queue_rows=25,
    # set the gap between resource icons - and the entities using them in resource use steps -
    # to be 30 units
    gap_between_resources=30,
    # set the height and width of the plot in pixels
    plotly_height=525,
    plotly_width=900,
    # set the duration of each frame in milliseconds
    frame_duration=200,
    # set the duration of the transition (movement phase) between each frame in milliseconds
    frame_transition_duration=600,
    # set the internal coordinate grid limits
    override_x_max=700,
    override_y_max=600,
    # stop the animation after the length of time we simulated for has elapsed
    limit_duration=RESULTS_COLLECTION_PERIOD,
    # if a queue reaches 25 entities, wrap it onto a new line so it doesn't just extend off the page
    wrap_queues_at=25,
    # if the number of resources exceeds 50, wrap them onto a new line
    # (not really needed in this example)
    wrap_resources_at=50,
    # if the number of people queueing in a step exceeds 75, replace the final entity with text
    # identifying the number of additional people in the queue
    step_snapshot_max=75,
    # specify the start time that will be used for time 0 in the time display
    start_time="12:00:00",
    # define the size of text layers in the animation
    text_size=20,
    # hide the layer of text that displays the 'label' field from the event positioning dataframe
    display_stage_labels=False,
    # add a background image created in the free draw.io tool that is a floor plan of the cafe
    add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_10_advanced_ciw/cafe_floorplan.drawio%20(1).png"
)